<a href="https://colab.research.google.com/github/Gaganpreet21kaur/cost-sensitive-fraud-detection/blob/main/fraud_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q xgboost shap imbalanced-learn

## 1. Environment

Colab provides a fresh environment for each session, so this cell must be run every time the notebook is opened.


# Cost-Sensitive Learning with XGBoost and SHAP for Optimising Fraud Detection Thresholds in Imbalanced Financial Transaction Data

MSc Thesis — implementation notebook

This notebook contains the full experimental pipeline for the study: data acquisition,
preprocessing, baseline and cost-sensitive model training, probability calibration,
cost-based threshold optimisation, SHAP explainability, statistical significance testing
and evaluation against pre-declared success criteria.

**Execution order.** Run every cell from top to bottom. Sections 1–11 define the pipeline
and produce no results on their own. Sections 12–18 run the experiments and write all
tables and figures to `results/` and `figures/`.

**Datasets.** The credit card data is downloaded directly from OpenML. PaySim is obtained from the Kaggle release, which the simulator's own repository identifies as the dataset distribution; Section 3 records the provenance and citation for each.

In [ ]:
import os
import glob
import json
import time
import platform
import subprocess
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import imblearn
import scipy

from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    precision_score, recall_score, f1_score, average_precision_score,
    roc_auc_score, brier_score_loss, precision_recall_curve, confusion_matrix,
)

from scipy import stats
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUTPUT_DIR = "fraud_detection_outputs"
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/fraud_detection_outputs"
    IN_COLAB = True
except Exception as exc:
    print("Google Drive not mounted (%s)." % exc)
    print("Output goes to a local folder and will NOT survive the session ending.")

RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
FIGURES_DIR = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

print("python      ", platform.python_version())
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("xgboost     ", xgb.__version__)
print("shap        ", shap.__version__)
print("output dir  ", OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
python       3.13.15
numpy        2.1.3
pandas       2.2.3
xgboost      3.4.1
shap         0.52.0
output dir   /content/drive/MyDrive/fraud_detection_outputs


## 2. Configuration and cost model

Two cost regimes are used.

Fixed cost regime. Every missed fraud carries the same penalty and every false alert carries the same penalty:

Total Cost = FN x C(FN) + FP x C(FP)

C(FN) is set to the mean value of fraudulent transactions in the training split of the dataset being analysed, so it is derived from the data rather than assumed. The test split takes no part in setting the costs.

C(FP) is an assumed administrative cost per false alert, covering manual review and the customer friction of a declined transaction. It is specified per dataset rather than as a single constant, because transaction values in the two datasets differ by roughly four orders of magnitude: mean fraud is 127.88 on the credit card data against approximately 1.35 million on PaySim. A review cost fixed at 4.37 would imply a cost ratio of about 309,000:1 on PaySim and drive the cost-minimising threshold to zero, which is an artefact of the scale mismatch rather than a property of the data. The review cost is therefore scaled with each dataset so that the implied ratio C(FN)/C(FP) is approximately 29:1 in both cases, and the two datasets are compared under the same cost structure. Section 13 reports a sensitivity analysis across ratios from 2:1 to 250:1, so no conclusion depends on the exact value chosen.

Instance-dependent regime. Following the example-dependent cost framework of Bahnsen et al. (2014), the cost of a missed fraud is the amount of that specific transaction:

| | Predicted genuine | Predicted fraud |
| Actual fraud | Amount(i) | C(TP) |
| Actual genuine | 0 | C(FP) |

This regime is what makes the objective genuinely financial: a missed transfer of 5,000 counts five times as heavily as a missed transfer of 1,000.

Cost of a correct alert. C(TP) is set to 0, so total cost is FN x C(FN) + FP x C(FP), matching the formula stated in the problem statement. Setting C(TP) equal to C(FP) instead gives the Bahnsen form, in which an analyst reviews every alert whether or not it turns out to be fraud. That variant is defensible and arguably more realistic, but it adds a third term to the cost equation and every reported figure would have to be re-quoted against it, so it is not the default here.

Success criteria. The thresholds in SUCCESS_CRITERIA were declared before the experimental runs and are reported unchanged in Section 15, whether met or not.

Theoretical check. Elkan (2001) gives the cost-minimising threshold in closed form as (C(FP) - C(TN)) / (C(FP) - C(TN) + C(FN) - C(TP)). With calibrated probabilities the threshold found by search should sit close to this value; a large gap between the two indicates the scores are not behaving as probabilities. Both numbers are printed for every fixed-cost experiment so the comparison can be reported.

In [ ]:
CONFIG = {
    "random_state": RANDOM_STATE,
    "test_size": 0.20,
    "val_size": 0.20,
    "c_tp": 0.00,            # cost charged on a correct alert (see Section 2)
    "n_threshold_candidates": 4000,
    "bootstrap_reps": 1000,
    "shap_sample_size": 4000,
    "smote_max_train_rows": 1_000_000,
    "mlp_max_train_rows": 400_000,
    "neg_subsample": 1.0,    # keep all genuine transactions; lower this only if RAM-bound
    "use_calibration": True,
}


C_FP_BY_DATASET = {
    "creditcard": 4.37,
    "paysim": 46209.63,
}
CONFIG["c_fp"] = C_FP_BY_DATASET["creditcard"]   # overwritten per experiment

XGB_PARAMS = {
    "n_estimators": 600,
    "max_depth": 5,
    "learning_rate": 0.05,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "min_child_weight": 1,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "eval_metric": "aucpr",
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
}

# Success criteria declared before the experimental runs and reported unchanged in
# Section 15 whether met or not. A criterion that is not met is reported as not met.
SUCCESS_CRITERIA = {
    "cost_reduction_pct": 20.0,
    "recall": 0.70,
    "value_recall": 0.70,
    "alert_rate": 0.02,
    "auprc": 0.60,
    "mcnemar_p": 0.05,
}

PAYSIM_CSV_PATH = os.path.join(OUTPUT_DIR, "data", "PS_20174392719_1491204439457_log.csv")

## 3. Data

### 3.1 European credit card dataset — OpenML dataset 1597

Downloaded directly from OpenML, the repository cited in the proposal, using sklearn.datasets.fetch_openml. No account or API token is required. The data were released by the Machine Learning Group at Universite Libre de Bruxelles and are described in Dal Pozzolo et al. (2015).

This copy has 284,807 rows and 30 columns: V1–V28 (PCA components), Amount and Class. Of these, 29 are used as features. It does not carry the Time column found on mirror sites, because OpenML designates Time a row identifier rather than a predictive attribute, so no hour-of-day feature is derived for this dataset. Amount is unscaled, which is what the instance-dependent cost model requires. 492 transactions (0.1727%) are fraudulent, giving an imbalance ratio of approximately 578:1.

### 3.2 PaySim mobile money dataset

The proposal cites github.com/EdgarLopezPhD/PaySim. That repository holds the Java source of the simulator only; the 6.3 million-row CSV used throughout the PaySim literature is generated output distributed elsewhere, and the repository README points to the Kaggle release for it. The released file is therefore used, and cited as Lopez-Rojas et al. (2016) for the origin of the data together with the distribution from which the file was obtained. This is the same file used by the PaySim studies in the literature review, so the results remain comparable with published work.

The file contains 6,362,620 transactions of which 8,213 (0.1291%) are fraudulent, an imbalance ratio of approximately 774:1. Mean fraudulent transaction value is roughly 1.47 million, four orders of magnitude above the credit card data, which is why the review cost is specified per dataset in Section 2.

Fourteen features are used after preparation. Three are engineered: errorBalanceOrig and errorBalanceDest, which measure whether the origin and destination balances reconcile against the transaction amount, and hourOfDay, derived from the simulator's step counter. destIsMerchant flags merchant destinations. Transaction type is one-hot encoded.

Three columns are dropped at load time. nameOrig and nameDest are account identifiers, which a model can memorise rather than generalise from. isFlaggedFraud is the output of the simulator's own rule-based control and is therefore target leakage.

Unlike the credit card data, PaySim features are named and interpretable, which is what Research Question 2 requires and which PCA components cannot provide.

A loader for a third dataset, the simulated transactions published with Reproducible Machine Learning for Credit Card Fraud Detection (Le Borgne et al., 2022), remains in the code as an alternative but was not used in the reported experiments.

In [ ]:
DATA_CACHE = {}


def _finalise(frame, feature_cols, label_col, amount_col, order_col, name):
    X = frame[feature_cols].astype("float32").reset_index(drop=True)
    y = frame[label_col].astype(int).to_numpy()
    amount = frame[amount_col].astype(float).to_numpy()
    order = frame[order_col].astype(float).to_numpy() if order_col is not None else None
    return {"name": name, "X": X, "y": y, "amount": amount, "order": order}


def load_creditcard():
    frame = fetch_openml(data_id=1597, as_frame=True).frame
    frame["Class"] = frame["Class"].astype(int)
    features = [c for c in frame.columns if c != "Class"]
    return _finalise(frame, features, "Class", "Amount", None, "creditcard")


def load_paysim(path=None):
    path = path or PAYSIM_CSV_PATH
    if not os.path.exists(path):
        raise FileNotFoundError(
            "PaySim CSV not found at %s. Place the file there." % path)
    frame = pd.read_csv(path)
    frame.columns = [c.strip() for c in frame.columns]


    frame["errorBalanceOrig"] = (
        frame["oldbalanceOrg"] - frame["amount"] - frame["newbalanceOrig"])
    frame["errorBalanceDest"] = (
        frame["oldbalanceDest"] + frame["amount"] - frame["newbalanceDest"])
    frame["hourOfDay"] = frame["step"] % 24
    frame["destIsMerchant"] = frame["nameDest"].astype(str).str.startswith("M").astype(int)

    type_dummies = pd.get_dummies(frame["type"], prefix="type").astype(int)
    frame = pd.concat([frame, type_dummies], axis=1)

    features = ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest",
                "newbalanceDest", "errorBalanceOrig", "errorBalanceDest",
                "hourOfDay", "destIsMerchant"] + list(type_dummies.columns)
    return _finalise(frame, features, "isFraud", "amount", "step", "paysim")


LOADERS = {
    "creditcard": load_creditcard,
    "paysim": load_paysim,
}


def get_dataset(name):
    if name not in DATA_CACHE:
        started = time.time()
        data = LOADERS[name]()
        n_fraud = int(data["y"].sum())
        n = len(data["y"])
        data["summary"] = {
            "dataset": name,
            "rows": n,
            "features": data["X"].shape[1],
            "frauds": n_fraud,
            "fraud_rate_pct": round(100.0 * n_fraud / n, 4),
            "imbalance_ratio": round((n - n_fraud) / max(n_fraud, 1), 1),
            "mean_fraud_amount": round(float(data["amount"][data["y"] == 1].mean()), 2),
            "total_fraud_value": round(float(data["amount"][data["y"] == 1].sum()), 2),
            "load_seconds": round(time.time() - started, 1),
        }
        DATA_CACHE[name] = data
    return DATA_CACHE[name]

In [ ]:
SECOND_DATASET = "paysim"

profiles = []
for ds in ["creditcard", SECOND_DATASET]:
    try:
        profiles.append(get_dataset(ds)["summary"])
    except FileNotFoundError:
        print(f"{ds}: file not found - skipping. Experiments 3 and 4 unavailable.\n")
    except Exception as exc:
        print(f"{ds}: failed to load ({type(exc).__name__}: {exc})\n")

profile_df = pd.DataFrame(profiles)
profile_df.to_csv(os.path.join(RESULTS_DIR, "dataset_profile.csv"), index=False)
print(profile_df.to_string(index=False))

   dataset    rows  features  frauds  fraud_rate_pct  imbalance_ratio  mean_fraud_amount  total_fraud_value  load_seconds
creditcard  284807        29     492          0.1727            577.9             122.21       6.012797e+04          26.3
    paysim 6362620        14    8213          0.1291            773.7         1467967.30       1.205642e+10          27.6


## 4. Splitting and leakage control

The data are divided into training (60%), validation (20%) and test (20%) partitions. The validation partition is split again into two halves, used for the cross-fitted calibration described in Section 8; both halves then contribute to selecting the decision threshold. The test partition is used once, for reporting, and takes no part in training, calibration or threshold selection.

Splitting is chronological for PaySim, which carries a step index recording the hour of each transaction, so the model is always evaluated on transactions occurring after those it was trained on. The OpenML copy of the credit card data has no time column, so a stratified random split is used there and the fraud rate is preserved in every partition.

The chronological split has a consequence that must be declared. PaySim fraud is not distributed evenly across the simulated period, so the partitions do not share a common fraud rate: the training partition contains 3,189 frauds at a rate of 0.0835%, while the test partition contains 4,254 at 0.3343%, four times higher. Slightly more than half of all fraudulent transactions in the dataset fall in the final 20% of the simulated timeline. A stratified split would equalise the rates but would allow the model to train on transactions occurring after those it is tested on, which is not defensible for a fraud detection system intended for deployment. The chronological split is retained for that reason, and the difference in base rate is taken into account when interpreting the PaySim results in Chapter 4.

Leakage controls: no resampling, scaling or calibration is fitted on anything other than the training or calibration partitions; SMOTE is applied to training rows only; identifier columns and the simulator's own fraud flag are dropped at load time.

A further consequence of dataset size should be recorded. SMOTE is applied only where the training partition contains at most 1,000,000 rows, and the neural network is trained only where it contains at most 400,000, both limits being set by available memory. The PaySim training partition contains 3,817,572 rows, so neither the SMOTE nor the neural network comparison is available for that dataset. The model set therefore differs between the two datasets, and the comparisons in Chapter 4 are made within each dataset rather than across them.

In [ ]:
def make_splits(data, cfg=CONFIG):
    y = data["y"]
    n = len(y)
    rng = np.random.RandomState(cfg["random_state"])
    idx = np.arange(n)

    if data["order"] is None:
        train_idx, hold_idx = train_test_split(
            idx, test_size=cfg["test_size"] + cfg["val_size"],
            stratify=y, random_state=cfg["random_state"])
        rel = cfg["test_size"] / (cfg["test_size"] + cfg["val_size"])
        val_idx, test_idx = train_test_split(
            hold_idx, test_size=rel, stratify=y[hold_idx],
            random_state=cfg["random_state"])
        cal_idx, thr_idx = train_test_split(
            val_idx, test_size=0.5, stratify=y[val_idx],
            random_state=cfg["random_state"])
    else:
        # Chronological split. PaySim's step index counts hours, so roughly 8,500
        # transactions share each value. Each boundary is therefore moved back to the
        # start of its time index value, so transactions from the same hour are never
        # divided between two partitions.
        ordered = idx[np.argsort(data["order"], kind="mergesort")]
        order_sorted = np.asarray(data["order"], dtype=float)[ordered]
        n_train = int(n * (1 - cfg["test_size"] - cfg["val_size"]))
        n_val = int(n * cfg["val_size"])
        cut1 = int(np.searchsorted(order_sorted, order_sorted[n_train], side="left"))
        cut2 = int(np.searchsorted(
            order_sorted, order_sorted[min(n_train + n_val, n - 1)], side="left"))
        cut2 = max(cut2, cut1 + 1)
        train_idx = ordered[:cut1]
        val_idx = ordered[cut1:cut2]
        test_idx = ordered[cut2:]
        half = len(val_idx) // 2
        cal_idx, thr_idx = val_idx[:half], val_idx[half:]

    if cfg["neg_subsample"] < 1.0:
        genuine = train_idx[y[train_idx] == 0]
        fraud = train_idx[y[train_idx] == 1]
        keep = rng.choice(genuine, int(len(genuine) * cfg["neg_subsample"]), replace=False)
        train_idx = np.sort(np.concatenate([keep, fraud]))

    return {"train": train_idx, "cal": cal_idx, "thr": thr_idx, "test": test_idx}


SPLIT_LABELS = {"train": "train", "cal": "validation A", "thr": "validation B",
                "test": "test"}


def describe_splits(data, splits):
    rows = []
    for part in ["train", "cal", "thr", "test"]:
        sel = splits[part]
        rows.append({
            "partition": SPLIT_LABELS[part],
            "rows": len(sel),
            "frauds": int(data["y"][sel].sum()),
            "fraud_rate_pct": round(100.0 * data["y"][sel].mean(), 4),
            "fraud_value": round(float(data["amount"][sel][data["y"][sel] == 1].sum()), 2),
        })
    return pd.DataFrame(rows)

## 5. Cost model and evaluation metrics

cost_alert is the cost incurred if a transaction is flagged, cost_pass the cost incurred if it is allowed through. Total cost at a threshold is then the sum of cost_alert over flagged rows plus cost_pass over the rest, which covers both the fixed and the instance-dependent regime with the same code.

The threshold search is vectorised rather than looped. Scores are sorted once, cumulative sums of the two cost vectors are formed, and the total cost at any threshold is read off with a binary search. This makes a 4,000-point search over 1.27 million test transactions inexpensive, which matters because the search is repeated for every model and again for every ratio in the sensitivity analysis of Section 13.

Candidate thresholds combine 4,000 quantiles of the predicted score distribution with a uniform grid from 0.001 to 0.999, plus the conventional 0.5. The quantiles matter more than the grid: at a fraud rate below 0.2% the cost minimum can lie below 0.01, where a uniform grid has almost no resolution.

Alongside the standard classification metrics, two measures are reported that matter commercially and are not visible in precision or recall: value recall, the share of fraudulent money caught rather than the share of fraudulent transactions caught, and the alert rate, the proportion of all transactions sent for review, which is what determines whether an operating point is affordable to staff.

Both measures are reported because precision and recall can be identical at two operating points with very different commercial consequences. A model that catches 80% of fraudulent transactions but misses the largest ones has a high recall and a low value recall, and the second figure is the one that appears in the loss account.

In [ ]:
def make_cost_vectors(y, amount, instance_dependent, c_fn_fixed, cfg=CONFIG):
    y = np.asarray(y)
    cost_fn = np.asarray(amount, dtype=float) if instance_dependent \
        else np.full(len(y), float(c_fn_fixed))
    # Flagging a fraud costs C(TP), which is zero here; flagging a genuine transaction
    # costs C(FP). Passing a fraud costs C(FN); passing a genuine transaction is free.
    cost_alert = np.where(y == 1, cfg["c_tp"], cfg["c_fp"]).astype(float)
    cost_pass = np.where(y == 1, cost_fn, 0.0)
    return cost_alert, cost_pass


def cost_curve(p, cost_alert, cost_pass, thresholds):
    order = np.argsort(p, kind="mergesort")
    p_sorted = p[order]
    cum_alert = np.concatenate([[0.0], np.cumsum(cost_alert[order])])
    cum_pass = np.concatenate([[0.0], np.cumsum(cost_pass[order])])
    cut = np.searchsorted(p_sorted, thresholds, side="left")
    return cum_pass[cut] + (cum_alert[-1] - cum_alert[cut])


def candidate_thresholds(p, cfg=CONFIG):
    quantiles = np.quantile(p, np.linspace(0.0, 1.0, cfg["n_threshold_candidates"]))
    grid = np.linspace(0.001, 0.999, 999)
    return np.unique(np.concatenate([quantiles, grid, [0.5]]))


def optimal_threshold(p, cost_alert, cost_pass, cfg=CONFIG):
    grid = candidate_thresholds(p, cfg)
    costs = cost_curve(p, cost_alert, cost_pass, grid)
    best = int(np.argmin(costs))
    return float(grid[best]), grid, costs


def evaluate(y, p, threshold, cost_alert, cost_pass, amount, model_name):
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    total = float(cost_alert[pred == 1].sum() + cost_pass[pred == 0].sum())
    fraud_value = float(amount[y == 1].sum())
    caught_value = float(amount[(y == 1) & (pred == 1)].sum())
    return {
        "model": model_name,
        "threshold": round(float(threshold), 6),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "auprc": average_precision_score(y, p),
        "roc_auc": roc_auc_score(y, p),
        "value_recall": caught_value / fraud_value if fraud_value else np.nan,
        "alert_rate": float(pred.mean()),
        "total_cost": total,
        "cost_per_1k_tx": 1000.0 * total / len(y),
    }

## 6. Models

Four XGBoost strategies are compared, plus logistic regression and a neural network.

**Baseline XGBoost** — no imbalance handling, evaluated at the conventional 0.5 cut-off. All cost reductions are measured against this.

**Cost-sensitive XGBoost** — scale_pos_weight = C(FN)/C(FP), so missed frauds dominate the loss.

**Instance-dependent XGBoost** — per-row sample_weight, set to the transaction amount for frauds and the review cost for genuine transactions, so the loss follows the money at stake. Weights are normalised to a mean of one. This arm runs only under the instance-dependent cost regime.

**SMOTE + XGBoost** — an ablation, not part of the proposed method. It tests whether oversampling and cost weighting correct the same imbalance twice. Applied to training rows only.

Logistic regression gives a linear reference point, and the multi-layer perceptron provides the neural network comparison requested in the proposal feedback. Both are wrapped in a pipeline with a StandardScaler so scaling is fitted on training rows only. XGBoost is scale-invariant and needs none.

All XGBoost arms share the same hyperparameters, so the only difference between the baseline and the cost-sensitive arms is the cost treatment, not model capacity.

Two arms are unavailable on PaySim. SMOTE runs only where the training partition holds at most 1,000,000 rows and the neural network only up to 400,000, both limits set by memory. PaySim's training partition holds 3.8 million rows, so neither appears in its results, and comparisons are made within each dataset rather than across them.

In [ ]:
def fit_xgb(X, y, params=None, scale_pos_weight=None, sample_weight=None):
    settings = dict(XGB_PARAMS)
    if params:
        settings.update(params)
    if scale_pos_weight is not None:
        settings["scale_pos_weight"] = float(scale_pos_weight)
    model = xgb.XGBClassifier(**settings)
    model.fit(X, y, sample_weight=sample_weight)
    return model


def fit_logreg(X, y):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ])
    model.fit(X, y)
    return model


def fit_mlp(X, y):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu",
                              alpha=1e-4, batch_size=1024, learning_rate_init=1e-3,
                              max_iter=60, early_stopping=True, n_iter_no_change=5,
                              random_state=RANDOM_STATE)),
    ])
    model.fit(X, y)
    return model


def fit_smote_xgb(X, y, params=None):
    sampler = SMOTE(sampling_strategy=1.0, k_neighbors=5, random_state=RANDOM_STATE)
    X_res, y_res = sampler.fit_resample(X, y)
    return fit_xgb(X_res, y_res, params=params)

##7. Hyperparameter search

A randomised search over the XGBoost hyperparameter space, scored by average precision (AUPRC) under stratified 3-fold cross-validation on the training partition only. AUPRC is used rather than accuracy or AUC-ROC because it is least distorted by extreme class imbalance.

RandomizedSearchCV is used instead of an exhaustive grid because SEARCH_SPACE holds 8,640 combinations. Twenty draws are sampled. Above 200,000 training rows the search runs on a subsample that keeps every fraud.

The search is disabled by default, so all reported results use the fixed hyperparameters in XGB_PARAMS. Set RUN_TUNING = True to run it; selected parameters are saved to results/tuned_params_*.json and reused automatically.

Because no arm is tuned, the baseline and cost-sensitive arms are compared at identical capacity, so the difference between them comes from the cost treatment alone.

The reported figures are therefore untuned and likely conservative. The search is also scored by AUPRC rather than by cost, which is inconsistent with the study's objective; a cost-scored search is noted as future work.

In [ ]:
RUN_TUNING = False

SEARCH_SPACE = {
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.02, 0.05, 0.1, 0.2],
    "n_estimators": [200, 400, 600],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 5, 20],
    "reg_lambda": [0.5, 1.0, 5.0],
}


def tune_hyperparameters(X, y, dataset_name, n_iter=20, max_rows=200_000):
    # Checked before the cache, so that RUN_TUNING = False always means untuned
    # parameters regardless of what is left in the results folder.
    if not RUN_TUNING:
        return {}

    path = os.path.join(RESULTS_DIR, "tuned_params_%s.json" % dataset_name)
    if os.path.exists(path):
        with open(path) as fh:
            return json.load(fh)

    n_pos = int((y == 1).sum())
    if len(y) > max_rows and max_rows > n_pos:
        keep_pos = np.where(y == 1)[0]
        keep_neg = np.random.RandomState(RANDOM_STATE).choice(
            np.where(y == 0)[0], max_rows - n_pos, replace=False)
        sel = np.sort(np.concatenate([keep_pos, keep_neg]))
        X, y = X.iloc[sel], y[sel]

    base = dict(XGB_PARAMS)
    for key in SEARCH_SPACE:
        base.pop(key, None)
    search = RandomizedSearchCV(
        xgb.XGBClassifier(**base),
        SEARCH_SPACE,
        n_iter=n_iter,
        scoring="average_precision",
        cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbose=1,
    )
    search.fit(X, y)
    with open(path, "w") as fh:
        json.dump(search.best_params_, fh, indent=2)
    print("best AUPRC %.4f" % search.best_score_)
    print(search.best_params_)
    return search.best_params_

## 8. Probability calibration

A cost-optimal threshold is only meaningful if the score behaves like a probability, and gradient boosting on heavily imbalanced data does not produce well-calibrated scores by default. Isotonic regression is used, fitted on validation data only.

The validation partition is calibrated by two-fold cross-fitting: a calibrator fitted on half A is applied to half B and vice versa, so every validation transaction receives a calibrated score that was not fitted on itself. The threshold is then selected on the whole validation partition rather than on half of it, which matters because the credit card data contains only 492 frauds in total and a threshold chosen on 50 of them is unstable. A single calibrator fitted on the full validation partition is applied to the test partition, which takes no part in either step.

Brier scores before and after calibration are reported so the effect is documented rather than assumed.

Isotonic regression is monotonic, so it cannot change the ranking of transactions and therefore leaves AUPRC and AUC-ROC unaffected. What it changes is where a given probability value sits, and so which threshold is optimal. It does introduce ties, since the fitted function is piecewise constant.

Calibration can be disabled with  `CONFIG["use_calibration"] = False`, which makes the threshold search run on the raw model scores. The uncalibrated optimum will generally sit some distance from the Elkan value quoted in Section 2, and that gap then needs explaining.



In [ ]:
def cross_fitted_calibration(p_a, y_a, p_b, y_b, p_test):
    if not CONFIG["use_calibration"]:
        return np.concatenate([p_a, p_b]), p_test
    iso_a = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(p_a, y_a)
    iso_b = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(p_b, y_b)
    p_val = np.concatenate([iso_b.predict(p_a), iso_a.predict(p_b)])
    iso_full = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(
        np.concatenate([p_a, p_b]), np.concatenate([y_a, y_b]))
    return p_val, iso_full.predict(p_test)


def calibration_report(y_true, p_raw, p_cal, model_name):
    return {
        "model": model_name,
        "brier_raw": brier_score_loss(y_true, p_raw),
        "brier_calibrated": brier_score_loss(y_true, p_cal),
        "mean_predicted_raw": float(p_raw.mean()),
        "mean_predicted_calibrated": float(p_cal.mean()),
        "observed_fraud_rate": float(y_true.mean()),
    }

## 9. Statistical testing

McNemar's test with continuity correction compares the baseline and the cost-sensitive
model on the test partition, using the discordant pairs b and c:

    chi-squared = (|b - c| - 1)^2 / (b + c)

The exact binomial form is used when b + c is small, where the chi-squared approximation
is unreliable. A bootstrap over test transactions gives a confidence interval for the cost
difference, since the total cost is dominated by a small number of high-value frauds and a
point estimate alone would overstate the precision of the result.

In [ ]:
def mcnemar_test(y_true, pred_a, pred_b):
    correct_a = pred_a == y_true
    correct_b = pred_b == y_true
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    if b + c == 0:
        return {"b": b, "c": c, "statistic": np.nan, "p_value": 1.0, "test": "none"}
    if b + c < 25:
        p = stats.binomtest(min(b, c), b + c, 0.5).pvalue
        return {"b": b, "c": c, "statistic": np.nan, "p_value": float(p), "test": "exact"}
    chi2 = (abs(b - c) - 1.0) ** 2 / (b + c)
    return {"b": b, "c": c, "statistic": float(chi2),
            "p_value": float(stats.chi2.sf(chi2, 1)), "test": "chi2"}


def bootstrap_cost_difference(cost_a, cost_b, reps=None, seed=RANDOM_STATE):
    reps = reps or CONFIG["bootstrap_reps"]
    diff = cost_a - cost_b
    rng = np.random.RandomState(seed)
    n = len(diff)
    means = np.empty(reps)
    for i in range(reps):
        means[i] = diff[rng.randint(0, n, n)].mean()
    return {
        "mean_saving_per_tx": float(diff.mean()),
        "ci_low": float(np.percentile(means, 2.5)),
        "ci_high": float(np.percentile(means, 97.5)),
    }


def per_row_cost(y, p, threshold, cost_alert, cost_pass):
    pred = (p >= threshold).astype(int)
    return np.where(pred == 1, cost_alert, cost_pass)

## 10. Figures

Every figure is written to figures/ as a 200 dpi PNG named after the experiment that produced it. The cost-against-threshold curve is the central figure of the results chapter: it shows the total cost at every possible cut-off, with the conventional 0.5 marked against the cost-minimising value.

Six figures are produced per experiment: class balance by count and by value, total cost against threshold, the decomposition of that cost into missed fraud and alert handling, the calibration curve before and after isotonic regression, precision-recall curves for every model, and paired confusion matrices for the baseline and the proposed model.

The y-axis is truncated on the two cost figures. Total cost rises steeply as the threshold approaches zero, because every transaction is then alerted, and on an untruncated axis the operating region near the minimum is compressed into a flat line. The truncation is noted on the axis label of each affected figure.

In [ ]:
def save_fig(fig, name):
    path = os.path.join(FIGURES_DIR, name + ".png")
    fig.savefig(path)
    plt.close(fig)
    return path


def plot_class_balance(y, amount, tag):
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
    counts = [int((y == 0).sum()), int((y == 1).sum())]
    axes[0].bar(["Genuine", "Fraud"], counts, color=["#4c72b0", "#c44e52"])
    axes[0].set_yscale("log")
    axes[0].set_ylabel("Transactions (log scale)")
    axes[0].set_title("Class distribution (%.3f%% fraud)" % (100 * y.mean()))
    for i, v in enumerate(counts):
        axes[0].text(i, v, "{:,}".format(v), ha="center", va="bottom")

    values = [float(amount[y == 0].sum()), float(amount[y == 1].sum())]
    axes[1].bar(["Genuine", "Fraud"], values, color=["#4c72b0", "#c44e52"])
    axes[1].set_yscale("log")
    axes[1].set_ylabel("Transaction value (log scale)")
    axes[1].set_title("Value at stake by class")
    fig.tight_layout()
    return save_fig(fig, "class_balance_" + tag)


def plot_cost_curve(grid, costs, t_star, tag, title):
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.plot(grid, costs, color="#4c72b0", lw=1.6)
    cost_half = float(np.interp(0.5, grid, costs))
    cost_star = float(costs[np.argmin(costs)])
    ax.axvline(0.5, color="#8c8c8c", ls="--", lw=1.2,
               label="Default 0.5 (cost %s)" % format(cost_half, ",.0f"))
    ax.axvline(t_star, color="#c44e52", ls="-", lw=1.4,
               label="Optimal %.4f (cost %s)" % (t_star, format(cost_star, ",.0f")))
    # Costs explode as the threshold approaches zero, when every transaction is alerted.
    # The y-axis is truncated so the operating region remains readable.
    ax.set_ylim(0, 1.6 * max(cost_half, cost_star))
    ax.set_xlabel("Decision threshold")
    ax.set_ylabel("Total expected cost on validation split (axis truncated)")
    ax.set_title(title)
    ax.legend(loc="upper center", fontsize=8)
    fig.tight_layout()
    return save_fig(fig, "cost_vs_threshold_" + tag)


def plot_cost_decomposition(y, p, cost_alert, cost_pass, grid, tag, title):
    order = np.argsort(p, kind="mergesort")
    p_sorted = p[order]
    cum_alert = np.concatenate([[0.0], np.cumsum(cost_alert[order])])
    cum_pass = np.concatenate([[0.0], np.cumsum(cost_pass[order])])
    cut = np.searchsorted(p_sorted, grid, side="left")
    missed = cum_pass[cut]
    alerts = cum_alert[-1] - cum_alert[cut]

    total = missed + alerts
    best = grid[int(np.argmin(total))]

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.plot(grid, missed, label="Cost of missed fraud", color="#c44e52")
    ax.plot(grid, alerts, label="Cost of alerts raised", color="#4c72b0")
    ax.plot(grid, total, label="Total cost", color="#333333", lw=1.8)
    ax.axvline(best, color="#333333", ls=":", lw=1.2,
               label="Cost-optimal %.4f" % best)
    ax.axvline(0.5, color="#8c8c8c", ls="--", lw=1.0, label="Default 0.5")
    ax.set_xscale("log")
    ax.set_xlim(max(grid[grid > 0].min(), 1e-5), 1.0)
    if total.min() > 0:
        ax.set_ylim(0, 2.0 * float(total.min()))
    ax.set_xlabel("Decision threshold (log scale)")
    ax.set_ylabel("Cost on validation split (axis truncated)")
    ax.set_title(title)
    ax.legend(fontsize=8)
    fig.tight_layout()
    return save_fig(fig, "cost_decomposition_" + tag)


def plot_pr_curve(y, curves, tag, title):
    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    for name, p in curves.items():
        precision, recall, _ = precision_recall_curve(y, p)
        ax.plot(recall, precision, lw=1.4,
                label="%s (AUPRC %.3f)" % (name, average_precision_score(y, p)))
    ax.axhline(y.mean(), color="#8c8c8c", ls="--", lw=1.0, label="No-skill")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(title)
    ax.legend(fontsize=8)
    fig.tight_layout()
    return save_fig(fig, "pr_curve_" + tag)


def plot_confusion_pair(y, pairs, tag, title):
    fig, axes = plt.subplots(1, len(pairs), figsize=(4.4 * len(pairs), 3.6))
    axes = np.atleast_1d(axes)
    for ax, (name, pred) in zip(axes, pairs.items()):
        cm = confusion_matrix(y, pred, labels=[0, 1])
        ax.imshow(np.log1p(cm), cmap="Blues")
        ax.set_xticks([0, 1], ["Genuine", "Fraud"])
        ax.set_yticks([0, 1], ["Genuine", "Fraud"])
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        ax.set_title(name, fontsize=9)
        ax.grid(False)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, "{:,}".format(cm[i, j]), ha="center", va="center",
                        color="#c44e52", fontsize=10)
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    return save_fig(fig, "confusion_" + tag)


def plot_calibration(y, p_raw, p_cal, tag, title, bins=12):
    # Log axes on both sides: at a fraud rate below 0.2% the predicted and observed
    # rates span several orders of magnitude, and on a linear axis the calibrated
    # series is compressed into the corner of the plot.
    fig, ax = plt.subplots(figsize=(5.4, 4.4))
    lo = np.inf
    for label, probs, colour in [("Raw", p_raw, "#4c72b0"),
                                 ("Isotonic", p_cal, "#c44e52")]:
        edges = np.unique(np.quantile(probs, np.linspace(0, 1, bins + 1)))
        if len(edges) < 2:
            continue
        idx = np.clip(np.searchsorted(edges, probs, side="right") - 1, 0, len(edges) - 2)
        xs, ys = [], []
        for b in range(len(edges) - 1):
            m = idx == b
            if m.sum() > 0 and probs[m].mean() > 0 and y[m].mean() > 0:
                xs.append(probs[m].mean())
                ys.append(y[m].mean())
        if xs:
            ax.plot(xs, ys, marker="o", ms=3, lw=1.2, color=colour, label=label)
            lo = min(lo, min(xs), min(ys))

    lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
    lo = max(lo if np.isfinite(lo) else 1e-6, 1e-6)
    ax.plot([lo, lim], [lo, lim], ls="--", color="#8c8c8c", lw=1.0,
            label="Perfect calibration")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Mean predicted probability (log scale)")
    ax.set_ylabel("Observed fraud rate (log scale)")
    ax.set_title(title)
    ax.legend(fontsize=8)
    fig.tight_layout()
    return save_fig(fig, "calibration_" + tag)

## 11. Experiment runner

`run_experiment` executes one complete experiment: split, train every model, calibrate, select each model's cost-minimising threshold on the validation split, evaluate once on the test split, run the significance tests and write the tables and figures. Cost reduction is always measured against the baseline XGBoost at the 0.5 cut-off, which is the conventional operating point and therefore the reference system.

Every model is evaluated at two operating points: the default 0.5 and its own cost-minimising threshold. This gives the factorial comparison the study requires, since the baseline at its cost-optimal threshold isolates the effect of moving the cut-off, and the cost-sensitive model at 0.5 isolates the effect of reweighting the loss.

A further reference row, "No model (approve everything)", records the cost of taking no action at all. It bounds the problem: any model whose cost exceeds this figure is worse than having no fraud detection.

The model designated as the proposed system depends on the cost regime. Under fixed costs it is the cost-sensitive XGBoost; under instance-dependent costs it is the instance-dependent XGBoost. This is the model carried into the significance tests, the SHAP analysis and the success criteria.

In [ ]:
def run_experiment(dataset_name, instance_dependent=False,
                   run_smote=True, run_mlp=True, verbose=True):
    tag = "%s_%s" % (dataset_name, "instance" if instance_dependent else "fixed")
    started = time.time()

    data = get_dataset(dataset_name)
    splits = make_splits(data)
    X, y, amount = data["X"], data["y"], data["amount"]

    X_tr, y_tr, amt_tr = X.iloc[splits["train"]], y[splits["train"]], amount[splits["train"]]
    X_cal, y_cal = X.iloc[splits["cal"]], y[splits["cal"]]
    X_thr, y_thr, amt_thr = X.iloc[splits["thr"]], y[splits["thr"]], amount[splits["thr"]]
    X_te, y_te, amt_te = X.iloc[splits["test"]], y[splits["test"]], amount[splits["test"]]

    y_val = np.concatenate([y_cal, y_thr])
    amt_val = np.concatenate([amount[splits["cal"]], amt_thr])

    # The review cost is scaled to the value scale of the dataset, so that the ratio
    # C(FN)/C(FP) is comparable across datasets rather than being an artefact of the
    # currency units. See Section 2.
    CONFIG["c_fp"] = C_FP_BY_DATASET.get(dataset_name, CONFIG["c_fp"])

    c_fn_fixed = float(amt_tr[y_tr == 1].mean())
    cost_ratio = c_fn_fixed / CONFIG["c_fp"]

    ca_val, cp_val = make_cost_vectors(y_val, amt_val, instance_dependent, c_fn_fixed)
    ca_te, cp_te = make_cost_vectors(y_te, amt_te, instance_dependent, c_fn_fixed)

    if verbose:
        print("=" * 96)
        print("Experiment: %s | cost regime: %s"
              % (tag, "instance-dependent" if instance_dependent else "fixed"))
        print("C(FP) = %.2f, C(TP) = %.2f, C(FN) = %.2f (mean fraud amount in training "
              "split), ratio = %.1f"
              % (CONFIG["c_fp"], CONFIG["c_tp"], c_fn_fixed, cost_ratio))
        if not instance_dependent:
            elkan = CONFIG["c_fp"] / (CONFIG["c_fp"] + c_fn_fixed - CONFIG["c_tp"])
            print("Elkan closed-form threshold for this cost matrix: %.4f" % elkan)
        print("=" * 96)
        print(describe_splits(data, splits).to_string(index=False))
        print()

    params = tune_hyperparameters(X_tr, y_tr, dataset_name)

    fitted = {}
    fitted["XGBoost (baseline)"] = fit_xgb(X_tr, y_tr, params=params)
    fitted["XGBoost (cost-sensitive)"] = fit_xgb(
        X_tr, y_tr, params=params, scale_pos_weight=cost_ratio)

    if instance_dependent:
        weights = np.where(y_tr == 1, amt_tr, CONFIG["c_fp"]).astype(float)
        weights = weights / weights.mean()
        fitted["XGBoost (instance-dependent)"] = fit_xgb(
            X_tr, y_tr, params=params, sample_weight=weights)

    if run_smote and len(y_tr) <= CONFIG["smote_max_train_rows"]:
        fitted["SMOTE + XGBoost"] = fit_smote_xgb(X_tr, y_tr, params=params)
    fitted["Logistic regression"] = fit_logreg(X_tr, y_tr)
    if run_mlp and len(y_tr) <= CONFIG["mlp_max_train_rows"]:
        fitted["Neural network (MLP)"] = fit_mlp(X_tr, y_tr)

    rows = []
    calibration_rows = []
    probabilities = {}
    val_probabilities = {}
    raw_probabilities = {}
    thresholds = {}
    curves = {}

    rows.append({
        "model": "No model (approve everything)", "threshold": np.nan,
        "TP": 0, "FP": 0, "FN": int(y_te.sum()), "TN": int((y_te == 0).sum()),
        "precision": 0.0, "recall": 0.0, "f1": 0.0, "auprc": np.nan, "roc_auc": np.nan,
        "value_recall": 0.0, "alert_rate": 0.0,
        "total_cost": float(cp_te.sum()),
        "cost_per_1k_tx": 1000.0 * float(cp_te.sum()) / len(y_te),
    })

    for name, model in fitted.items():
        p_cal_raw = model.predict_proba(X_cal)[:, 1]
        p_thr_raw = model.predict_proba(X_thr)[:, 1]
        p_te_raw = model.predict_proba(X_te)[:, 1]

        p_val, p_te = cross_fitted_calibration(
            p_cal_raw, y_cal, p_thr_raw, y_thr, p_te_raw)
        calibration_rows.append(calibration_report(y_te, p_te_raw, p_te, name))

        t_star, grid, costs = optimal_threshold(p_val, ca_val, cp_val)
        thresholds[name] = t_star
        probabilities[name] = p_te.astype("float32")
        val_probabilities[name] = p_val.astype("float32")
        raw_probabilities[name] = p_te_raw.astype("float32")
        curves[name] = (grid, costs)

        rows.append(evaluate(y_te, p_te, 0.5, ca_te, cp_te, amt_te, name + " @0.5"))
        rows.append(evaluate(y_te, p_te, t_star, ca_te, cp_te, amt_te,
                             name + " @cost-optimal"))

    comparison = pd.DataFrame(rows)
    reference = float(
        comparison.loc[comparison["model"] == "XGBoost (baseline) @0.5", "total_cost"].iloc[0])
    comparison["cost_reduction_pct"] = 100.0 * (reference - comparison["total_cost"]) / reference
    comparison["experiment"] = tag

    headline = "XGBoost (instance-dependent)" if instance_dependent \
        else "XGBoost (cost-sensitive)"
    baseline_pred = (probabilities["XGBoost (baseline)"] >= 0.5).astype(int)
    headline_pred = (probabilities[headline] >= thresholds[headline]).astype(int)

    stats_result = mcnemar_test(y_te, baseline_pred, headline_pred)
    stats_result.update(bootstrap_cost_difference(
        per_row_cost(y_te, probabilities["XGBoost (baseline)"], 0.5, ca_te, cp_te),
        per_row_cost(y_te, probabilities[headline], thresholds[headline], ca_te, cp_te),
    ))
    stats_result.update({"experiment": tag, "baseline": "XGBoost (baseline) @0.5",
                         "compared_with": headline + " @cost-optimal"})

    calibration_df = pd.DataFrame(calibration_rows)
    comparison.to_csv(os.path.join(RESULTS_DIR, "comparison_%s.csv" % tag), index=False)
    calibration_df.to_csv(os.path.join(RESULTS_DIR, "calibration_%s.csv" % tag), index=False)
    describe_splits(data, splits).to_csv(
        os.path.join(RESULTS_DIR, "splits_%s.csv" % tag), index=False)

    plot_class_balance(y, amount, dataset_name)
    grid_h, costs_h = curves[headline]
    plot_cost_curve(grid_h, costs_h, thresholds[headline], tag,
                    "%s: total cost against decision threshold" % headline)
    plot_cost_decomposition(
        y_val, val_probabilities[headline].astype(float), ca_val, cp_val, grid_h, tag,
        "Where the cost comes from (%s)" % tag)
    plot_calibration(y_te, raw_probabilities[headline], probabilities[headline], tag,
                     "Calibration on test split (%s)" % headline)
    plot_pr_curve(y_te, probabilities, tag,
                  "Precision-recall on test split (%s)" % tag)
    plot_confusion_pair(
        y_te,
        {"XGBoost baseline @0.5": baseline_pred,
         headline + " @cost-optimal": headline_pred},
        tag, "Confusion matrices (%s)" % tag)

    # Stratified SHAP sample that over-represents fraud. Frauds are drawn at random
    # rather than taken in order, because on a chronologically split dataset the first
    # n frauds all come from the earliest part of the test window.
    rng = np.random.RandomState(RANDOM_STATE)
    n_sample = min(CONFIG["shap_sample_size"], len(y_te))
    fraud_idx = np.where(y_te == 1)[0]
    genuine_idx = np.where(y_te == 0)[0]
    n_fraud_take = min(len(fraud_idx), n_sample // 2)
    take_fraud = rng.choice(fraud_idx, n_fraud_take, replace=False)
    n_genuine_take = min(len(genuine_idx), n_sample - n_fraud_take)
    take_genuine = rng.choice(genuine_idx, n_genuine_take, replace=False)
    shap_idx = np.sort(np.concatenate([take_fraud, take_genuine]))

    result = {
        "tag": tag,
        "dataset": dataset_name,
        "instance_dependent": instance_dependent,
        "comparison": comparison,
        "calibration": calibration_df,
        "statistics": stats_result,
        "thresholds": thresholds,
        "headline": headline,
        "models": fitted,
        "c_fp": CONFIG["c_fp"],
        "c_fn_fixed": c_fn_fixed,
        "cost_ratio": cost_ratio,
        "shap_X": X_te.iloc[shap_idx].copy(),
        "shap_y": y_te[shap_idx],
        "shap_amount": amt_te[shap_idx],
        "arrays": {
            "y_val": y_val, "amt_val": amt_val,
            "y_test": y_te, "amt_test": amt_te,
            "p_val": val_probabilities, "p_test": probabilities,
        },
        "runtime_seconds": round(time.time() - started, 1),
    }

    if verbose:
        display_cols = ["model", "threshold", "TP", "FP", "FN", "precision", "recall",
                        "f1", "auprc", "value_recall", "alert_rate", "total_cost",
                        "cost_reduction_pct"]
        print(comparison[display_cols].round(4).to_string(index=False))
        print()
        print("McNemar: b=%s c=%s p=%.3g (%s)" % (
            stats_result["b"], stats_result["c"], stats_result["p_value"],
            stats_result["test"]))
        print("Bootstrap mean saving per transaction: %.4f (95%% CI %.4f to %.4f)" % (
            stats_result["mean_saving_per_tx"], stats_result["ci_low"],
            stats_result["ci_high"]))
        print("Runtime: %.1f s" % result["runtime_seconds"])
    return result


EXPERIMENTS = {}

## 12. Experiments

Four experiments: two datasets, each under both cost regimes. Experiments 1 and 2 use the credit card data, which is the standard benchmark and keeps the results comparable with published work. Experiments 3 and 4 use PaySim, whose named features are what Research Question 2 depends on.

Each experiment reports every model at two operating points, the default 0.5 and its own cost-minimising threshold, so the contribution of moving the threshold can be separated from the contribution of reweighting the loss.

### Experiment 1 — credit card, fixed cost per fraud

In [ ]:
EXPERIMENTS["cc_fixed"] = run_experiment("creditcard", instance_dependent=False)

Experiment: creditcard_fixed | cost regime: fixed
C(FP) = 4.37, C(TP) = 0.00, C(FN) = 127.88 (mean fraud amount in training split), ratio = 29.3
Elkan closed-form threshold for this cost matrix: 0.0330
   partition   rows  frauds  fraud_rate_pct  fraud_value
       train 170884     295          0.1726     37724.02
validation A  28480      49          0.1721      8184.52
validation B  28481      49          0.1720      4345.08
        test  56962      99          0.1738      9874.35

                                 model  threshold  TP  FP  FN  precision  recall     f1  auprc  value_recall  alert_rate  total_cost  cost_reduction_pct
         No model (approve everything)        NaN   0   0  99     0.0000  0.0000 0.0000    NaN        0.0000      0.0000  12659.9254           -417.3308
               XGBoost (baseline) @0.5      0.500  80   4  19     0.9524  0.8081 0.8743 0.8399        0.7399      0.0015   2447.1626              0.0000
      XGBoost (baseline) @cost-optimal      0.059  85

### Experiment 2 — credit card, cost of a missed fraud equal to the amount at risk

In [ ]:
EXPERIMENTS["cc_instance"] = run_experiment("creditcard", instance_dependent=True)

Experiment: creditcard_instance | cost regime: instance-dependent
C(FP) = 4.37, C(TP) = 0.00, C(FN) = 127.88 (mean fraud amount in training split), ratio = 29.3
   partition   rows  frauds  fraud_rate_pct  fraud_value
       train 170884     295          0.1726     37724.02
validation A  28480      49          0.1721      8184.52
validation B  28481      49          0.1720      4345.08
        test  56962      99          0.1738      9874.35

                                     model  threshold  TP  FP  FN  precision  recall     f1  auprc  value_recall  alert_rate  total_cost  cost_reduction_pct
             No model (approve everything)        NaN   0   0  99     0.0000  0.0000 0.0000    NaN        0.0000      0.0000     9874.35           -281.8860
                   XGBoost (baseline) @0.5      0.500  80   4  19     0.9524  0.8081 0.8743 0.8399        0.7399      0.0015     2585.68              0.0000
          XGBoost (baseline) @cost-optimal      0.059  85  67  14     0.5592  0.85

### Experiment 3 — PaySim, fixed cost per fraud

This dataset is larger. If the session runs out of memory, set `CONFIG["neg_subsample"]`to 0.3 and re-run: every fraudulent transaction is kept and only genuine transactions are sampled. Any such subsampling must be stated in the methodology.

Two features of this dataset shape the result and should be read alongside it. The split is chronological, and PaySim fraud is concentrated late in the simulated period, so the test partition carries a fraud rate several times that of the training partition. And the training partition exceeds the row limits set in Section 2, so neither the SMOTE nor the neural network arm is available here.

In [ ]:
import os
print("PaySim present:", os.path.exists(PAYSIM_CSV_PATH))
print(os.listdir(os.path.dirname(PAYSIM_CSV_PATH)))

PaySim present: True
['PS_20174392719_1491204439457_log.csv']


In [ ]:
try:
    EXPERIMENTS["second_fixed"] = run_experiment(SECOND_DATASET, instance_dependent=False)
except FileNotFoundError as e:
    print("Skipped - PaySim data not available yet.")
    print(e)

Experiment: paysim_fixed | cost regime: fixed
C(FP) = 46209.63, C(TP) = 0.00, C(FN) = 1352100.83 (mean fraud amount in training split), ratio = 29.3
Elkan closed-form threshold for this cost matrix: 0.0330
   partition    rows  frauds  fraud_rate_pct  fraud_value
       train 3792821    3175          0.0837 4.292920e+09
validation A  638138     456          0.0715 6.515202e+08
validation B  638138     324          0.0508 4.093803e+08
        test 1293523    4258          0.3292 6.702595e+09

                                 model  threshold   TP   FP   FN  precision  recall     f1  auprc  value_recall  alert_rate   total_cost  cost_reduction_pct
         No model (approve everything)        NaN    0    0 4258     0.0000  0.0000 0.0000    NaN        0.0000      0.0000 5.757245e+09        -209223.0694
               XGBoost (baseline) @0.5      0.500 4256    1    2     0.9998  0.9995 0.9996 0.9995        0.9999      0.0033 2.750411e+06              0.0000
      XGBoost (baseline) @cost-o

### Experiment 4 — PaySim, cost of a missed fraud equal to the amount at risk

In [ ]:
try:
    EXPERIMENTS["second_instance"] = run_experiment(SECOND_DATASET, instance_dependent=True)
except FileNotFoundError as e:
    print("Skipped - PaySim data not available yet.")
    print(e)

Experiment: paysim_instance | cost regime: instance-dependent
C(FP) = 46209.63, C(TP) = 0.00, C(FN) = 1352100.83 (mean fraud amount in training split), ratio = 29.3
   partition    rows  frauds  fraud_rate_pct  fraud_value
       train 3792821    3175          0.0837 4.292920e+09
validation A  638138     456          0.0715 6.515202e+08
validation B  638138     324          0.0508 4.093803e+08
        test 1293523    4258          0.3292 6.702595e+09

                                     model  threshold   TP  FP   FN  precision  recall     f1  auprc  value_recall  alert_rate   total_cost  cost_reduction_pct
             No model (approve everything)        NaN    0   0 4258     0.0000  0.0000 0.0000    NaN        0.0000      0.0000 6.702595e+09        -838637.5946
                   XGBoost (baseline) @0.5      0.500 4256   1    2     0.9998  0.9995 0.9996 0.9995        0.9999      0.0033 7.991289e+05              0.0000
          XGBoost (baseline) @cost-optimal      0.520 4253   0  

## 13. Sensitivity to the cost ratio

`C(FN)` is derived from the training data but `C(FP)` is assumed, so the ratio between them is a modelling choice. This section re-optimises the threshold across ratios from 2:1 to 250:1 and reports how the operating point and the saving move, showing which conclusions hold across the range and which depend on the ratio used.

The analysis reuses the probabilities each experiment already produced rather than retraining, so only the cost vectors and the threshold change. It therefore tests the sensitivity of the decision rule, not of the trained model. Retraining at each ratio would be a stronger test and is noted as future work.

Only the fixed-cost experiments are covered, since the instance-dependent regime takes `C(FN)` from each transaction's amount and has no single ratio to vary.

In [ ]:
def cost_ratio_sensitivity(result, ratios=(2, 5, 10, 25, 50, 100, 250)):
    arrays = result["arrays"]
    headline = result["headline"]
    # The review cost belongs to the experiment that produced these probabilities.
    # CONFIG["c_fp"] holds whichever dataset ran last, so it is reset here before any
    # cost vectors are built - make_cost_vectors reads it internally for the alert cost.

    c_fp = result["c_fp"]
    CONFIG["c_fp"] = c_fp
    rows = []
    for ratio in ratios:
        c_fn = ratio * c_fp
        ca_val, cp_val = make_cost_vectors(arrays["y_val"], arrays["amt_val"], False, c_fn)
        ca_te, cp_te = make_cost_vectors(arrays["y_test"], arrays["amt_test"], False, c_fn)

        reference = evaluate(arrays["y_test"], arrays["p_test"]["XGBoost (baseline)"], 0.5,
                             ca_te, cp_te, arrays["amt_test"], "baseline")["total_cost"]
        t_star, _, _ = optimal_threshold(arrays["p_val"][headline], ca_val, cp_val)
        tuned = evaluate(arrays["y_test"], arrays["p_test"][headline], t_star,
                         ca_te, cp_te, arrays["amt_test"], headline)
        rows.append({
            "cost_ratio": ratio,
            "C_FP": c_fp,
            "C_FN": c_fn,
            "optimal_threshold": t_star,
            "precision": tuned["precision"],
            "recall": tuned["recall"],
            "alert_rate": tuned["alert_rate"],
            "value_recall": tuned["value_recall"],
            "cost_reduction_pct": 100.0 * (reference - tuned["total_cost"]) / reference,
        })
    return pd.DataFrame(rows)


def plot_sensitivity(frame, tag):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    axes[0].semilogx(frame["cost_ratio"], frame["optimal_threshold"], marker="o",
                     color="#4c72b0")
    axes[0].axhline(0.5, color="#8c8c8c", ls="--", lw=1.0)
    axes[0].set_xlabel("C(FN) / C(FP)")
    axes[0].set_ylabel("Cost-optimal threshold")
    axes[0].set_title("Operating point against cost ratio")

    axes[1].semilogx(frame["cost_ratio"], frame["recall"], marker="o",
                     label="Recall", color="#c44e52")
    axes[1].semilogx(frame["cost_ratio"], frame["alert_rate"], marker="s",
                     label="Alert rate", color="#4c72b0")
    axes[1].set_xlabel("C(FN) / C(FP)")
    axes[1].set_ylabel("Proportion")
    axes[1].set_title("Detection and workload against cost ratio")
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    return save_fig(fig, "cost_sensitivity_" + tag)


sensitivity_tables = {}
for key in ["cc_fixed", "second_fixed"]:
    if key in EXPERIMENTS:
        frame = cost_ratio_sensitivity(EXPERIMENTS[key])
        frame.insert(0, "experiment", EXPERIMENTS[key]["tag"])
        sensitivity_tables[key] = frame
        plot_sensitivity(frame, EXPERIMENTS[key]["tag"])
        frame.to_csv(os.path.join(
            RESULTS_DIR, "cost_sensitivity_%s.csv" % EXPERIMENTS[key]["tag"]), index=False)
        print(frame.round(4).to_string(index=False))
        print()

      experiment  cost_ratio  C_FP    C_FN  optimal_threshold  precision  recall  alert_rate  value_recall  cost_reduction_pct
creditcard_fixed           2  4.37    8.74             0.3480     0.9302  0.8081      0.0015        0.7222             -4.7619
creditcard_fixed           5  4.37   21.85             0.1130     0.7944  0.8586      0.0019        0.7461              7.0707
creditcard_fixed          10  4.37   43.70             0.1130     0.7944  0.8586      0.0019        0.7461             16.4948
creditcard_fixed          25  4.37  109.25             0.1130     0.7944  0.8586      0.0019        0.7461             22.3382
creditcard_fixed          50  4.37  218.50             0.1130     0.7944  0.8586      0.0019        0.7461             24.3187
creditcard_fixed         100  4.37  437.00             0.1130     0.7944  0.8586      0.0019        0.7461             25.3151
creditcard_fixed         250  4.37 1092.50             0.0053     0.7944  0.8586      0.0019        0.7461     

## 14. SHAP explainability
SHAP values are computed with `TreeExplainer`, which is exact for tree ensembles, on a stratified sample of the test partition that over-represents fraud so that the explanations reflect the minority class rather than the bulk of genuine traffic.

Four outputs are produced per experiment: a bar chart of mean absolute SHAP value, a beeswarm plot showing the direction of each feature's effect, a dependence plot for the strongest feature, and a waterfall plot decomposing the single highest-value fraud in the sample. The ranking table is written to `results/shap_importance_*.csv`.

The credit card features are PCA components, so its explanations can only be reported at the level of V1–V28 and `Amount`. Interpretation in terms of named behaviour — transaction type, balance movements — is possible only on PaySim, which is the reason two datasets are used.

SHAP is applied to the proposed model of each experiment, so the fixed and instance-dependent rankings can be compared directly. A feature that rises in the ranking when costs become value-dependent is one the cost model has made the classifier attend to.



In [ ]:
def shap_analysis(result, max_display=15):
    tag = result["tag"]
    model = result["models"][result["headline"]]
    X = result["shap_X"]
    y = result["shap_y"]
    amount = result["shap_amount"]

    explainer = shap.TreeExplainer(model)
    values = explainer(X)

    importance = pd.DataFrame({
        "feature": X.columns,
        "mean_abs_shap": np.abs(values.values).mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    importance.insert(0, "experiment", tag)
    importance.to_csv(os.path.join(RESULTS_DIR, "shap_importance_%s.csv" % tag), index=False)

    shap.plots.bar(values, max_display=max_display, show=False)
    save_fig(plt.gcf(), "shap_bar_" + tag)

    shap.plots.beeswarm(values, max_display=max_display, show=False)
    save_fig(plt.gcf(), "shap_beeswarm_" + tag)

    top = importance.loc[0, "feature"]
    shap.plots.scatter(values[:, top], show=False)
    save_fig(plt.gcf(), "shap_dependence_%s_%s" % (tag, top))

    costly = int(np.argmax(np.where(y == 1, amount, -np.inf)))
    shap.plots.waterfall(values[costly], max_display=max_display, show=False)
    save_fig(plt.gcf(), "shap_waterfall_" + tag)

    print("Top features for %s (%s)" % (tag, result["headline"]))
    print(importance.head(max_display).round(5).to_string(index=False))
    print()
    return importance


shap_tables = {key: shap_analysis(result) for key, result in EXPERIMENTS.items()}

Top features for creditcard_fixed (XGBoost (cost-sensitive))
      experiment feature  mean_abs_shap
creditcard_fixed     V14        1.88869
creditcard_fixed      V4        1.82545
creditcard_fixed     V12        0.78499
creditcard_fixed     V10        0.68286
creditcard_fixed      V7        0.38734
creditcard_fixed      V3        0.36809
creditcard_fixed     V16        0.36348
creditcard_fixed     V11        0.35706
creditcard_fixed  Amount        0.35594
creditcard_fixed      V8        0.32940
creditcard_fixed     V15        0.32706
creditcard_fixed     V26        0.32572
creditcard_fixed     V19        0.32550
creditcard_fixed      V2        0.30110
creditcard_fixed     V21        0.29807

Top features for creditcard_instance (XGBoost (instance-dependent))
         experiment feature  mean_abs_shap
creditcard_instance      V4        2.15500
creditcard_instance     V14        1.81101
creditcard_instance     V10        1.26478
creditcard_instance     V12        0.82035
creditcard_inst

## 15. Pre-declared success criteria

The thresholds in `SUCCESS_CRITERIA` were set in Section 2 before the experimental runs reported here. They are reported unchanged. A criterion that is not met is reported as not met and discussed in the thesis; it is not adjusted after the fact.

Six criteria are checked against the proposed model of each experiment at its cost-optimal threshold: cost reduction against the baseline, recall, value recall, alert rate, AUPRC, and the McNemar p-value.

In [ ]:
def check_success_criteria(result):
    comparison = result["comparison"]
    label = result["headline"] + " @cost-optimal"
    row = comparison.loc[comparison["model"] == label].iloc[0]
    stats_result = result["statistics"]

    checks = [
        ("Cost reduction vs baseline @0.5 (%)", row["cost_reduction_pct"],
         SUCCESS_CRITERIA["cost_reduction_pct"], "at least"),
        ("Recall", row["recall"], SUCCESS_CRITERIA["recall"], "at least"),
        ("Value recall", row["value_recall"], SUCCESS_CRITERIA["value_recall"], "at least"),
        ("Alert rate", row["alert_rate"], SUCCESS_CRITERIA["alert_rate"], "at most"),
        ("AUPRC", row["auprc"], SUCCESS_CRITERIA["auprc"], "at least"),
        ("McNemar p-value", stats_result["p_value"],
         SUCCESS_CRITERIA["mcnemar_p"], "at most"),
    ]

    rows = []
    for name, achieved, target, direction in checks:
        met = achieved >= target if direction == "at least" else achieved <= target
        rows.append({
            "experiment": result["tag"],
            "criterion": name,
            "target": "%s %s" % (direction, target),
            "achieved": round(float(achieved), 4),
            "met": bool(met),
        })
    return pd.DataFrame(rows)


criteria_frames = [check_success_criteria(r) for r in EXPERIMENTS.values()]
success_criteria = pd.concat(criteria_frames, ignore_index=True)
success_criteria.to_csv(os.path.join(RESULTS_DIR, "success_criteria.csv"), index=False)
print(success_criteria.to_string(index=False))

         experiment                           criterion        target  achieved   met
   creditcard_fixed Cost reduction vs baseline @0.5 (%) at least 20.0   22.9135  True
   creditcard_fixed                              Recall  at least 0.7    0.8586  True
   creditcard_fixed                        Value recall  at least 0.7    0.7461  True
   creditcard_fixed                          Alert rate  at most 0.02    0.0019  True
   creditcard_fixed                               AUPRC  at least 0.6    0.8514  True
   creditcard_fixed                     McNemar p-value  at most 0.05    0.0106  True
creditcard_instance Cost reduction vs baseline @0.5 (%) at least 20.0    1.3540 False
creditcard_instance                              Recall  at least 0.7    0.8586  True
creditcard_instance                        Value recall  at least 0.7    0.7461  True
creditcard_instance                          Alert rate  at most 0.02    0.0017  True
creditcard_instance                               AUPR

## 16. Consolidated results

Three tables are written. `all_results.csv `contains every operating point from every experiment. `significance_tests.csv` contains the McNemar and bootstrap results for each experiment. `headline_comparison.csv` contains the three rows that carry the argument: the baseline at 0.5, the baseline at its cost-optimal threshold, and the proposed model at its cost-optimal threshold.

The headline table is the one to reproduce in the results chapter. Comparing its three rows separates the effect of moving the threshold from the effect of reweighting the loss, which is the factorial comparison the study is built around.

In [ ]:
if not EXPERIMENTS:
    raise SystemExit("No experiments have been run. Run Section 12 first.")

all_results = pd.concat([r["comparison"] for r in EXPERIMENTS.values()], ignore_index=True)
all_stats = pd.DataFrame([r["statistics"] for r in EXPERIMENTS.values()])

all_results.to_csv(os.path.join(RESULTS_DIR, "all_results.csv"), index=False)
all_stats.to_csv(os.path.join(RESULTS_DIR, "significance_tests.csv"), index=False)

headline_rows = []
for result in EXPERIMENTS.values():
    comparison = result["comparison"]
    for label in ["XGBoost (baseline) @0.5",
                  "XGBoost (baseline) @cost-optimal",
                  result["headline"] + " @cost-optimal"]:
        match = comparison.loc[comparison["model"] == label]
        if len(match):
            headline_rows.append(match.iloc[0])

headline = pd.DataFrame(headline_rows)[
    ["experiment", "model", "threshold", "TP", "FP", "FN", "precision", "recall",
     "f1", "auprc", "value_recall", "alert_rate", "total_cost", "cost_reduction_pct"]]
headline.to_csv(os.path.join(RESULTS_DIR, "headline_comparison.csv"), index=False)

print(headline.round(4).to_string(index=False))
print()
print(all_stats.round(4).to_string(index=False))

         experiment                                      model  threshold   TP  FP  FN  precision  recall     f1  auprc  value_recall  alert_rate   total_cost  cost_reduction_pct
   creditcard_fixed                    XGBoost (baseline) @0.5      0.500   80   4  19     0.9524  0.8081 0.8743 0.8399        0.7399      0.0015 2.447163e+03              0.0000
   creditcard_fixed           XGBoost (baseline) @cost-optimal      0.059   85  67  14     0.5592  0.8586 0.6773 0.8399        0.7462      0.0027 2.083082e+03             14.8776
   creditcard_fixed     XGBoost (cost-sensitive) @cost-optimal      0.113   85  22  14     0.7944  0.8586 0.8252 0.8514        0.7461      0.0019 1.886432e+03             22.9135
creditcard_instance                    XGBoost (baseline) @0.5      0.500   80   4  19     0.9524  0.8081 0.8743 0.8399        0.7399      0.0015 2.585680e+03              0.0000
creditcard_instance           XGBoost (baseline) @cost-optimal      0.059   85  67  14     0.5592  0.8586

## 17. Environment record

The figures printed here belong in the Requirements and Resources section of the thesis, replacing the estimate given in the proposal. The recorded runtimes are for the reported runs on this hardware and will differ on other machines.

In [ ]:
def read_total_memory_gb():
    try:
        with open("/proc/meminfo") as fh:
            for line in fh:
                if line.startswith("MemTotal"):
                    return round(int(line.split()[1]) / 1024 ** 2, 1)
    except Exception:
        pass
    return None


environment = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "platform": platform.platform(),
    "python": platform.python_version(),
    "cpu_count": os.cpu_count(),
    "total_ram_gb": read_total_memory_gb(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgb.__version__,
    "shap": shap.__version__,
    "imbalanced_learn": imblearn.__version__,
    "scipy": scipy.__version__,
    "random_state": RANDOM_STATE,
    "runtimes_seconds": {k: v["runtime_seconds"] for k, v in EXPERIMENTS.items()},
}

with open(os.path.join(RESULTS_DIR, "environment.json"), "w") as fh:
    json.dump(environment, fh, indent=2)

print(json.dumps(environment, indent=2))

{
  "timestamp": "2026-09-01 11:25:45",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "python": "3.13.15",
  "cpu_count": 2,
  "total_ram_gb": 12.7,
  "numpy": "2.1.3",
  "pandas": "2.2.3",
  "scikit_learn": "1.6.1",
  "xgboost": "3.4.1",
  "shap": "0.52.0",
  "imbalanced_learn": "0.14.2",
  "scipy": "1.16.3",
  "random_state": 42,
  "runtimes_seconds": {
    "cc_fixed": 92.2,
    "cc_instance": 143.1,
    "second_fixed": 603.3,
    "second_instance": 826.1
  }
}


In [ ]:
archive = os.path.join(OUTPUT_DIR, "results_and_figures.zip")
subprocess.run(["zip", "-qr", archive, RESULTS_DIR, FIGURES_DIR], check=False)
print("Written:", archive)
print("Results files:", len(os.listdir(RESULTS_DIR)))
print("Figure files:", len(os.listdir(FIGURES_DIR)))

if IN_COLAB:
    from google.colab import files
    files.download(archive)
else:
    print("Archive written to:", archive)

Written: /content/drive/MyDrive/fraud_detection_outputs/results_and_figures.zip
Results files: 24
Figure files: 40


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 18. Limitations of this implementation

Points to carry into the discussion chapter.

1. `C(FP)` is an assumed cost, not measured from data. Section 13 therefore reports results across ratios from 2:1 to 250:1 rather than relying on one figure.
2. `C(FP)` is also scaled per dataset so the two are compared at a similar ratio. That keeps them comparable, but the PaySim figure of 46,209 is a scaling choice rather than an observed review cost.
3. The calibration and threshold-selection halves come from the same validation partition. They are disjoint, but on PaySim both precede the test period, so a deployed system would need periodic recalibration.
4. The credit card data has no time column in the OpenML release, so its split is random rather than chronological. A model split randomly across time-ordered transactions can learn from transactions that occur after those it is tested on.
5. PaySim fraud is concentrated late in the simulated period, so the chronological split gives the test partition a fraud rate about four times that of the training partition. The two partitions therefore differ in base rate as well as in time.
6. SMOTE and the neural network were not run on PaySim, because its training partition exceeds the row limits set in Section 2. The model set differs between datasets, so comparisons hold within a dataset rather than across them.
7. Reported results use fixed hyperparameters; the search in Section 7 was not run. No arm is tuned, so the comparison between them is fair, but the absolute figures are likely conservative.
8. Fraud labels are treated as immediately available. In production, labels arrive after an investigation delay, which is not modelled here.
9. SHAP values explain the model, not the underlying fraud. A feature with a high SHAP value is one the model relies on, which is not the same as a cause.